## Stage 4: Knowledge Graph-based Context Organization


### Import Necessary Packages and Define Global Variables


In [1]:
import pickle
import json
import networkx as nx
import torch
from collections import defaultdict
from FlagEmbedding import FlagReranker
from pathlib import Path
from tqdm import tqdm
from typing import Any


DATA_DIR: Path = Path("data")
KG_DATA: Path = DATA_DIR / "gmq.pickle"
CHUNK_DATA: Path = DATA_DIR / "expanded_chunks.json"
QUESTION_DATA: Path = DATA_DIR / "hotpot_dev_distractor_v1_sample100.json"
JSON_OUTPPUT: Path = DATA_DIR / "context_organized.json"
PICKLE_OUTPUT: Path = DATA_DIR / "umq.pickle"


#### Reranking Model


In [2]:
TORCH_DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
RERANKER: FlagReranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    query_max_length=256,
    passage_max_length=512,
    use_fp16=True,
    devices=[TORCH_DEVICE],
)  # Setting use_fp16 to True speeds up computation with a slight performance degradation


### Read Knowledge Graph and Data


In [3]:
with open(KG_DATA, "rb") as kg_pickle:
    expanded_kg: nx.Graph = pickle.load(kg_pickle)

with open(CHUNK_DATA, "r", encoding="utf-8") as chunk_json:
    raw_data_chunks: list[dict[str, Any]] = json.load(chunk_json)

with open(QUESTION_DATA, "r", encoding="utf-8") as question_json:
    raw_questions: list[dict[str, Any]] = json.load(question_json)

### Preprocessing


In [4]:
# Change to access chunks by ID
data_chunks: dict[str, dict[str, Any]] = {
    chunk["chunk_id"]: chunk for chunk in raw_data_chunks
}

for chunk in tqdm(data_chunks.values(), desc="Updating chunks"):
    chunk.pop("chunk_id")

# Same with question
questions: dict[str, dict[str, Any]] = {
    question["_id"]: question for question in raw_questions
}

for question in tqdm(questions.values(), desc="Updating questions"):
    question.pop("_id")
    question["graph"] = nx.Graph()

Updating questions: 100%|██████████| 100/100 [00:00<00:00, 385151.88it/s]


### Create Corresponding Subgraph for each Chunk


In [5]:
def similarity_function(question: str, source: str) -> float:
    """
    Calculate similarity between the query and the source document.
    Args:
        question (str): The query
        source (str): The source document/chunk

    Returns:
        similarity (float): The similarity between the provided arguments
    """
    return RERANKER.compute_score([(question, source)])[0]

In [6]:
for head, tail, data in tqdm(expanded_kg.edges(data=True), desc="Enumerating Edges"):
    chunk_info: dict[str, Any] = data_chunks[data["source_chunk_id"]]
    question_info: dict[str, Any] = questions[chunk_info["question_id"]]
    graph: nx.Graph = question_info["graph"]

    data["question_id"] = chunk_info["question_id"]
    data["weight"] = -similarity_function(
        question_info["question"], chunk_info["chunk_text"]
    )  # Negate similarity for MST

    graph.add_edge(head, tail, **data)

Enumerating Edges: 100%|██████████| 164/164 [04:00<00:00,  1.47s/it]


### Perform MST for All of the Subgraphs


In [10]:
umq: nx.Graph = nx.Graph()
context_organized: list[dict[str, str | list[str]]] = []


for question_id, question_info in tqdm(questions.items(), desc="Processing Subgraphs"):
    documents_candidates: list[str] = []
    question = question_info["question"]
    graph: nx.Graph = question_info["graph"]
    mst: nx.Graph = nx.minimum_spanning_tree(graph)
    if not mst:
        continue
    umq.add_edges_from(mst.edges)

    for subtree in nx.connected_components(mst):
        subgraph = graph.subgraph(subtree)
        min_weight_edge = min(subgraph.edges(data=True), key=lambda e: e[-1]["weight"])
        head, tail = min_weight_edge[:-1]

        dfs_order = nx.dfs_edges(
            subgraph,
            head,
            sort_neighbors=lambda v: sorted(v, key=lambda x: not x == tail),
        )
        added_chunks: set[str] = set()
        organized_chunks: str = ""

        for head, tail in dfs_order:
            edge = subgraph.get_edge_data(head, tail)
            chunk_id = edge["source_chunk_id"]

            if chunk_id not in added_chunks:
                organized_chunks = (
                    f"{organized_chunks}{data_chunks[chunk_id]['chunk_text']}\n"
                )
                added_chunks.add(chunk_id)

        documents_candidates.append(organized_chunks)

    documents_candidates: list[str] = sorted(
        documents_candidates, key=lambda x: -similarity_function(question, x)
    )

    context_organized.append(
        {
            "question_id": question_id,
            "question": question,
            "organized_document": documents_candidates,
        }
    )


Processing Subgraphs: 100%|██████████| 100/100 [00:37<00:00,  2.64it/s]


### Save the Results


In [11]:
with open(JSON_OUTPPUT, "w", encoding="utf-8") as umq_json_output:
    json.dump(context_organized, umq_json_output, indent=2)

with open(PICKLE_OUTPUT, "wb") as umq_pickle_output:
    pickle.dump(umq, umq_pickle_output)